# 融资压力原始来源核验，不训练
SOFR与准备金付息率使用官方来源，保留制度切换及修订信息；有效日期不是可用日期，时点资格完成前禁止作为预测输入。

In [1]:
from pathlib import Path
from datetime import datetime,timezone
import os,json,hashlib
import requests
import numpy as np
import pandas as pd
import qlib
from qlib.workflow import R
from src.data.qlib_native import build_qlib_store_from_frame,load_qlib_feature_frame,QlibProviderConfig
from src.data.nas_market_data import archive_nas_recorder
from src.data.minute_intake import atomic_json
root=Path.cwd()
while not (root/'SPEC.md').exists():root=root.parent
os.chdir(root);os.environ['MLFLOW_ALLOW_FILE_STORE']='true';card=root/'research/funding_spread_source_20261014';plan=json.loads((card/'plan.json').read_text());rawdir=root/plan['raw_directory'];rawdir.mkdir(parents=True,exist_ok=True);receipts={}
for series in ['IOER','IORB']:
 path=rawdir/(series+'.csv');rp=rawdir/(series+'_receipt.json')
 if not path.exists():
  response=requests.get('https://fred.stlouisfed.org/graph/fredgraph.csv',params={'id':series,'cosd':plan['start'],'coed':plan['as_of']},timeout=30);response.raise_for_status();path.write_bytes(response.content);atomic_json({'url':response.url,'http_status':response.status_code,'retrieved_at':datetime.now(timezone.utc).isoformat(),'sha256':hashlib.sha256(response.content).hexdigest()},rp)
 receipts[series]=json.loads(rp.read_text());assert hashlib.sha256(path.read_bytes()).hexdigest()==receipts[series]['sha256']
frames={}
for series in ['IOER','IORB']:
 data=pd.read_csv(rawdir/(series+'.csv'));dates=pd.to_datetime(data.iloc[:,0]);values=pd.to_numeric(data[series],errors='coerce');frames[series]=pd.Series(values.to_numpy(),index=dates)
 assert frames[series].index.is_unique and dates.le(pd.Timestamp(plan['as_of'])).all()
sofr_path=rawdir/'SOFR.json';receipt=json.loads((rawdir/'SOFR_receipt.json').read_text());assert hashlib.sha256(sofr_path.read_bytes()).hexdigest()==receipt['sha256'];receipts['SOFR']=receipt
rates=pd.DataFrame(json.loads(sofr_path.read_text())['refRates']).sort_values('effectiveDate');rates['datetime']=pd.to_datetime(rates.effectiveDate);assert rates.type.eq('SOFR').all() and rates.datetime.is_unique and rates.datetime.between(plan['start'],plan['as_of']).all()
rates=rates.set_index('datetime');policy=pd.Series(np.nan,index=rates.index);before=policy.index<pd.Timestamp('2021-07-29');policy.loc[before]=frames['IOER'].reindex(policy.index[before]);policy.loc[~before]=frames['IORB'].reindex(policy.index[~before]);assert policy.notna().all()
assert frames['IOER'].dropna().index.max()==pd.Timestamp('2021-07-28') and frames['IORB'].dropna().index.min()==pd.Timestamp('2021-07-29')
percentiles=rates[['percentPercentile1','percentPercentile25','percentPercentile75','percentPercentile99']].apply(pd.to_numeric,errors='coerce')
percentile_known=percentiles.notna().all(axis=1)
a=percentiles.loc[percentile_known];median=rates.loc[percentile_known,'percentRate']
assert (a.percentPercentile1<=a.percentPercentile25).all() and (a.percentPercentile25<=median).all() and (median<=a.percentPercentile75).all() and (a.percentPercentile75<=a.percentPercentile99).all()
frame=pd.DataFrame({'sofr_percent':rates.percentRate,'reserve_rate_percent':policy,'spread_bp':100*(rates.percentRate-policy),'reserve_rate_is_iorb':(~before).astype(float),'volume_billions':rates.volumeInBillions},index=rates.index);assert np.isfinite(frame).all().all()
frame['instrument']='USD_FUNDING_OBSERVATION';data=frame.reset_index();provider=root/plan['provider'];fields=[c for c in frame if c!='instrument'];manifest=build_qlib_store_from_frame(data,provider_uri=provider,freq='day',as_of=plan['as_of'],field_map={c:c for c in fields},source='Official NYFed SOFR and FRED Board reserve rates; EFFECTIVE-DATE observations, not point-in-time available features')
manifest.update(calendar_role='underlying_rate_effective_dates_not_publication_dates',forecast_input_qualified=False,requires_publication_revision_alignment=True);atomic_json(manifest,provider/'qlib_store_manifest.json')
qlib.init(provider_uri=str(provider),region='us');native=load_qlib_feature_frame(QlibProviderConfig(provider),instruments=['USD_FUNDING_OBSERVATION'],fields=['$'+c for c in fields],start=plan['start'],end=plan['as_of']).xs('USD_FUNDING_OBSERVATION',level='instrument');np.testing.assert_allclose(native.to_numpy(),frame[fields].to_numpy(),rtol=2e-7,atol=1e-4)
summary={'experiment':card.name,'decision':'raw_funding_source_collected_needs_historical_availability_alignment','new_fits':0,'new_backtests':0,'outcome_correlations':0,'objective_achieved':False,'rows':len(frame),'first_effective_date':str(frame.index.min().date()),'last_effective_date':str(frame.index.max().date()),'reserve_rate_missing_rows':int(policy.isna().sum()),'percentile_unknown_rows':int((~percentile_known).sum()),'revision_indicator_counts':rates.revisionIndicator.fillna('__unknown__').value_counts().to_dict(),'minimum_spread_bp':float(frame.spread_bp.min()),'maximum_spread_bp':float(frame.spread_bp.max()),'forecast_input_qualified':False,'first_vintage_certified':False,'provider_uri':str(provider),'region':'us','as_of':plan['as_of'],'qlib_docs':'https://qlib.readthedocs.io/en/latest/component/data.html','availability_notes':'SOFR describes previous business day transactions, normally published next business day08:00ET with possible same-day14:30 revisions. Never use effectiveDate as known date. Treasury and equity calendars differ.','economic_scope':'Observed repo-versus-administered-rate spread, not actual SOXL financing cost or a proven stock forecast','artifact_policy':{'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':['inputs/*','analysis/observations']}}
tracking=card/'qlib_recorder'
with R.start(experiment_name=card.name,recorder_name='funding_source_only',uri=str(tracking)):
 rec=R.get_recorder();rec.log_params(new_fits=0,new_backtests=0,forecast_input_qualified=False)
 rec.save_objects(artifact_path='inputs',plan=plan,receipts=receipts,sofr_payload=json.loads(sofr_path.read_text()),reserve_csvs={k:(rawdir/(k+'.csv')).read_bytes() for k in frames},provider_manifest=manifest)
 rec.save_objects(artifact_path='analysis',observations=native,raw_sofr=rates,summary=summary)
 artifacts=tracking/str(rec.experiment_id)/rec.id/'artifacts';rec.log_metrics(registered_artifact_count=sum(p.is_file() for p in artifacts.rglob('*')),source_rows=len(frame))
summary.update(recorder_id=rec.id,experiment_id=str(rec.experiment_id),recorder_uri=str(tracking));atomic_json({'recorder_id':rec.id,'entries':[{'path':str(p.relative_to(card)),'sha256':hashlib.sha256(p.read_bytes()).hexdigest()} for p in sorted(artifacts.rglob('*')) if p.is_file()]},card/'verification_manifest.json');summary['nas_evidence']=archive_nas_recorder(card/'nas_binding.json',workspace_root=root,card=card,recorder_id=rec.id,experiment_id=str(rec.experiment_id));atomic_json(summary,card/'summary.json');frame.groupby(frame.index.year).agg(observations=('spread_bp','size'),min_spread_bp=('spread_bp','min'),max_spread_bp=('spread_bp','max')).to_csv(card/'annual_source_range.csv')
import yaml
(card/'workflow.yaml').write_text(yaml.safe_dump({'experiment':card.name,'provider_uri':str(provider),'recorder_uri':str(tracking),'forecast_input_qualified':False,'artifact_policy':summary['artifact_policy']},sort_keys=False));print({k:summary[k] for k in ['rows','first_effective_date','last_effective_date','revision_indicator_counts','minimum_spread_bp','maximum_spread_bp']})


# 融资压力原始来源：已采集，尚不具备预测时点资格

本轮0拟合、0回测、0收益相关筛选。针对尚未在现有结论中找到的美元融资压力机制，取得NYFed官方SOFR及FRED所载美联储准备金付息率，独立保存载荷、取回时间、SHA256和原生Qlib存储；没有改变现役策略。

SOFR观察期2018-04-02至2026-10-09，共2130行。逐有效日期匹配准备金付息率，无缺口、不前向填充；2021-07-28以前用IOER，2021-07-29起用IORB，保留明确制度标记。两种制度的定义变化由美联储实施说明确认，不能将其静默当同一原始系列。利差统一为100*(SOFR百分数-准备金利率百分数)，单位bp，样本范围-21至315bp；没有按这个范围选择阈值。

2019-05-31和2021-08-05的SOFR分位数在官方载荷中为NA，但中央利率和成交量存在。保留这两日，不补分位数0、不删除利率。其余已知分位数的顺序检查通过。API当前全部revisionIndicator为空，只表示当前载荷标记为空，不等于历史首发版本已认证。

最重要的边界：SOFR有效日期是底层交易日期，通常下一业务日约08:00纽约时间公布，可能于同日约14:30修订。Treasury业务日与股票交易日不一致。当前provider明确calendar_role=underlying_rate_effective_dates_not_publication_dates、forecast_input_qualified=false，不得直接按有效日期与股票特征拼接，不得用今日下载的最终值冒充当日开盘前已知值。

决定：raw_funding_source_collected_needs_historical_availability_alignment。新增的是独立于股票OHLC的原始信息来源，还不是合格预测特征。融资压力到半导体收益或杠杆暴露的传导仅为待证机制；该利差不是SOXL实际融资费用，也不能直接转成现金规则。

下一步核对历史公布/修订日历与首版本或足够保守的可用时点；完成时点资格前不做收益检验。旧恢复分数和新低收敛的失败路线保持关闭。稳定净超额未实现。

主要来源：[NYFed公布与修订说明](https://www.newyorkfed.org/markets/reference-rates/additional-information-about-reference-rates)、[Markets API](https://markets.newyorkfed.org/static/docs/markets-api.html)、[美联储2021年实施说明](https://www.federalreserve.gov/newsevents/pressreleases/monetary20210728a1.htm)、[FRED IORB](https://fred.stlouisfed.org/series/IORB)、[FRED IOER](https://fred.stlouisfed.org/series/IOER)。

Recorder `86169eecce9a4e1eb1150dc77af64560`。[年度来源范围](annual_source_range.csv)。
